In [1]:
import hanoi
import numpy as np
from itertools import product
import os
import pandas as pd

In [2]:
M0 = np.array([0, 0.9, 1, 1.1, 2])
M1 = np.array([0, 0.9, 1, 1.1, 2])

In [3]:
V0 = 10 ** np.linspace(-2, 1, 11)
V1 = 10 ** np.linspace(-2, 1, 11)

In [4]:
parcombs = np.array(list(product(M0, M1, V0, V1)))

In [5]:
parcombs.shape

(3025, 4)

In [6]:
G = np.repeat([0,1,2], [25, 50, 25]).astype(float).reshape((1,100))

In [7]:
G

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 2., 2., 2., 2., 2.,
        2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2.,
        2., 2., 2., 2.]])

In [8]:
hanoi.fit_step?

Signature: hanoi.fit_step(boxes, z)
Docstring:
Computes fitness based on a step fitness function.

Arguments:
    z:     (N, n) array of n-dimensional phenotypes of N individuals.
    boxes: (m, 2, n) array of m boxes, where boxes[i, 0, :] = lower bounds, boxes[i, 1, :] = upper bounds of i-th box. 
           Fitness of phenotype within boxes is 1.

Returns: (N,) array of 0 or 1
File:      ~/SeaDrive/My Libraries/mylib/work/projects/python/packages/hanoi/hanoi/fitness.py
Type:      function

In [16]:
c = 0
out = []
for m0, m1, v0, v1 in parcombs:
    c+=1
    if c>100:
        break
    M_m = np.array([[m1 - m0]])
    M_v = np.array([[v1 - v0]])
    z_ref = np.array([m0])
    c_ref = np.array([[v0]])
    M = hanoi.MutEffect(mean = M_m, var = M_v, cov=np.array([[]]))
    res = hanoi.run_replicates(n_reps = 10, 
                               max_workers = 10,
                               n_gen = 0, 
                               genotype = G, 
                               mut_effect = M, 
                               record = False, 
                               mut_rate = 0.0, 
                               mean_0 = z_ref,
                               varcov_0 = c_ref, 
                               fit_func = "fit_step",
                               boxes = np.array([[[1], [np.inf]]])
                         )
    out.append([[m0, m1, v0, v1, res[j][0], res[j][1].allele_freqs_next()[0,0]] for j in range(len(res))])
out = np.array(out).reshape(-1, np.array(out).shape[2])
out_df = pd.DataFrame(out, columns = ["m0", "m1", "v0", "v1", "n_gen", "allele_fixed"])
out_df = out_df.astype({"n_gen": "int64", "allele_fixed": np.float64})

In [17]:
out_df

,m0,m1,v0,v1,n_gen,allele_fixed
0,0.0,0.0,0.010000,0.01,1,NaN
1,0.0,0.0,0.010000,0.01,1,NaN
2,0.0,0.0,0.010000,0.01,1,NaN
3,0.0,0.0,0.010000,0.01,1,NaN
4,0.0,0.0,0.010000,0.01,1,NaN
...,...,...,...,...,...,...
995,0.0,0.0,5.011872,0.01,11,0.0
996,0.0,0.0,5.011872,0.01,12,0.0
997,0.0,0.0,5.011872,0.01,7,0.0
998,0.0,0.0,5.011872,0.01,12,0.0


In [18]:
out_df.to_csv("output.txt", sep=" ", index=False, na_rep="NaN")

[[1, <hanoi.hanoi.Generation at 0x7805f8b76000>],
 [1, <hanoi.hanoi.Generation at 0x7805f8b767e0>],
 [1, <hanoi.hanoi.Generation at 0x7805f8b3a840>],
 [1, <hanoi.hanoi.Generation at 0x7805f8b74200>],
 [1, <hanoi.hanoi.Generation at 0x7805f8b75550>],
 [1, <hanoi.hanoi.Generation at 0x7805f8b741d0>],
 [1, <hanoi.hanoi.Generation at 0x7805f8b77350>],
 [1, <hanoi.hanoi.Generation at 0x7805f8b3bf20>],
 [1, <hanoi.hanoi.Generation at 0x7805f8b3b650>],
 [1, <hanoi.hanoi.Generation at 0x7805f8b3b470>]]

In [11]:
G

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 2., 2., 2., 2., 2.,
        2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2.,
        2., 2., 2., 2.]])

In [21]:
sim = hanoi.sim_generations(n_gen = 1, 
                      genotype = G, 
                      mut_effect = M, 
                      record = True,
                      mut_rate = 0.0, 
                      mean_0 = z_ref,
                      varcov_0 = c_ref, 
                      fit_func = "fit_neutral",
                      boxes = np.array([[[1], [np.inf]]])
                     )

In [23]:
sim.n_gen

1

In [14]:
sim.genotype

AttributeError: 'list' object has no attribute 'genotype'

In [18]:
sim.genotype_next

array([[[0, 2, 1, 0, 1, 0, 1, 0, 0, 2, 2, 0, 1, 2, 0, 1, 1, 1, 1, 0, 1,
         0, 1, 0, 1, 1, 1, 1, 2, 0, 1, 0, 1, 2, 1, 2, 1, 2, 1, 2, 1, 1,
         2, 1, 2, 2, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 2, 1, 1, 1, 1,
         2, 1, 2, 2, 0, 0, 1, 2, 1, 0, 1, 1, 2, 1, 1, 1, 2, 2, 1, 2, 0,
         2, 0, 0, 0, 2, 1, 0, 2, 2, 0, 1, 0, 1, 1, 1, 1]],

       [[1, 0, 2, 2, 2, 0, 2, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 2, 1, 2, 2,
         2, 1, 1, 0, 0, 0, 0, 0, 0, 1, 2, 1, 2, 1, 2, 1, 0, 1, 0, 1, 1,
         1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 2, 0, 0,
         1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 0, 1, 2, 1, 2, 0, 2, 2, 1, 1, 1,
         1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 2, 1]],

       [[1, 0, 0, 1, 2, 0, 0, 0, 0, 1, 2, 0, 2, 1, 1, 0, 1, 1, 0, 1, 0,
         1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 2, 2, 2, 0, 1, 1, 2,
         1, 2, 1, 2, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0,
         2, 0, 1, 2, 0, 0, 1, 1, 2, 1, 2, 1, 0, 1, 0, 2, 0, 0, 0, 2, 0,
         1, 1, 0

In [19]:
sim.n_gen

10

In [34]:
V0, V1 = np.meshgrid(v0, v1)

In [25]:
V0

array([[0, 1, 2],
       [0, 1, 2],
       [0, 1, 2]])

In [27]:
V0.flatten()

array([0, 1, 2, 0, 1, 2, 0, 1, 2])

In [28]:
V1.flatten()

array([0, 0, 0, 1, 1, 1, 2, 2, 2])

In [30]:
np.column_stack((V0.flatten(), V1.flatten()))

array([[0, 0],
       [1, 0],
       [2, 0],
       [0, 1],
       [1, 1],
       [2, 1],
       [0, 2],
       [1, 2],
       [2, 2]])

In [25]:
m0, m1, v0, v1 = [0, 0, 0.01, 1.2589254117941675]
M_m = np.array([[m1 - m0]])
M_v = np.array([[v1 - v0]])
z_ref = np.array([m0])
c_ref = np.array([[v0]])
M = hanoi.MutEffect(mean = M_m, var = M_v, cov=np.array([[]]))
res = hanoi.run_replicates(n_reps = 1, 
                           max_workers = 10,
                           n_gen = 0, 
                           genotype = G, 
                           mut_effect = M, 
                           record = False, 
                           mut_rate = 0.0, 
                           mean_0 = z_ref,
                           varcov_0 = c_ref, 
                           fit_func = "fit_step",
                           boxes = np.array([[[1], [np.inf]]])
                     )

In [26]:
res

[[4, <hanoi.hanoi.Generation at 0x77d604d4fc50>]]

In [139]:
v1

1.5

In [9]:
?hanoi.fit_multimodal

Signature: hanoi.fit_multimodal(sigma, p, z_opt, z)
Docstring:
Computes fitness based on a multimodal fitness function.

Arguments:
    sigma:  (n_peaks,) array specifying SD of n_peak peaks
    p:      (n_peaks,) array specifying relative height of peaks
    z_opt:  (n_peaks, n) array specifying the coordinate of n_peak peaks in the n-dimensional phenotype space
    z:      (N, n) array representing n-dimensional phenotypes of N individuals.
Returns: (N,) array representing fitness of N individuals
File:      ~/SeaDrive/My Libraries/mylib/work/projects/python/packages/hanoi/hanoi/fitness.py
Type:      function